## Topic: RunnableBranch

### Agenda 
- 1. Introduction of RunnableBranch

- 2. Practical Example of RunnableBranch

### 1. Introduction of RunnableBranch
- Definition:
    - RunnableBranch is a control flow component in LangChain that allows to conditionally route input data to different chains or runnables based on custom logic.

    - RunnableBranch dynamically chooses which Runnable should execute based on the input.

    - It Functions like an if-elif-else for chains, where we define a set of conditions functions, each associated with a runnable(eg. LLM call, prompt chain, or tools). the first matching condition is executed. if no condition matches, a default runnable is used(if provided)

In [ ]:
""" 
- Syntax of RunnableBrach:

RunnableBrach(
    (condition : Runnable), # execute when condition is true
    (condition: Runnable )  
    (Default Runnable)     # else execute this block, when condition is False 

)


"""

In [ ]:
"""  - Workflow of RunnableBranch

                         INPUT
                           │
                           ↓
                   RunnableBranch
                           │
                     Evaluate Conditions
                           │
             ┌─────────────┼─────────────┐
             ↓             ↓             ↓
         Condition 1   Condition 2    Default
             │             │             │
           True?         True?          Else
             │             │             │
             ↓             ↓             ↓
          Chain A       Chain B       Chain C
             │             │             │
             └─────────────┼─────────────┘
                           ↓
                       Selected
                       Output

Branch selection happens at runtime based on the actual input.
"""

### 2. Practical Example of RunnableBranch

In [ ]:
# Example 1: RunnableBranch
# purpose: topic -> prompt -> LLm -> parse --> RunnableParallel --> 
#                                                   -> 1. if words > 300 -> summaries(LLm) -> parse -> output
#                                                    -> 2. else words < 300 -> output
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
from langchain.schema.runnable import RunnableSequence, RunnableParallel, RunnablePassthrough, RunnableBranch, RunnableLambda

load_dotenv()

# Prompt 1: for report 
prompt1 = PromptTemplate(
    template='Write a detailed report on {topic}',
    input_variables=['topic']
)

# Prompt2: for summarize when LLm1 output is greater than 300 words
prompt2 = PromptTemplate(
    template='Summarize the following text \n {text}',
    input_variables=['text']
)

# Define model
model = ChatOpenAI()

# Define parse
parser = StrOutputParser()

# report generation chain
report_gen_chain = prompt1 | model | parser

# RunnableBranch 
# 1. if words > 300 -> summaries(LLm) -> parse -> output
# 2. else words < 300 -> output
branch_chain = RunnableBranch(
    (lambda x: len(x.split())>300, prompt2 | model | parser),
    RunnablePassthrough() # default runnable 
)

# Final chain -> Connection between report generation chain and brach chain
final_chain = RunnableSequence(report_gen_chain, branch_chain)

print(final_chain.invoke({'topic':'Russia vs Ukraine'}))



In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


technical_prompt = ChatPromptTemplate.from_template(
    """
    You are a technical AI instructor.

    Answer the following question with technical accuracy:

    {question}
    """
)


general_prompt = ChatPromptTemplate.from_template(
    """
    You are a helpful assistant.

    Answer the following question in simple language:

    {question}
    """
)

branch = RunnableBranch(
    (is_technical, technical_chain),
    general_chain
)


result = branch.invoke({
    "question": "What is a neural network?"
})

print(result)

In [ ]:
from langchain_core.runnables import RunnableLambda, RunnableBranch


# ---------------------------------------------------------
# Step 1: Define the functions for each branch
# ---------------------------------------------------------

def positive(number):
    return "The number is positive."


def negative(number):
    return "The number is negative."


def zero(number):
    return "The number is zero."


# ---------------------------------------------------------
# Step 2: Define the conditions
# ---------------------------------------------------------

is_positive = lambda x: x > 0

is_negative = lambda x: x < 0


# ---------------------------------------------------------
# Step 3: Convert functions into Runnables
# ---------------------------------------------------------

positive_runnable = RunnableLambda(positive)

negative_runnable = RunnableLambda(negative)

zero_runnable = RunnableLambda(zero)


# ---------------------------------------------------------
# Step 4: Create RunnableBranch
# ---------------------------------------------------------

branch = RunnableBranch(
    (is_positive, positive_runnable),
    (is_negative, negative_runnable),
    zero_runnable
)


# ---------------------------------------------------------
# Step 5: Test the branch
# ---------------------------------------------------------

print(branch.invoke(10))

print(branch.invoke(-5))

print(branch.invoke(0))

### Final Takeaway

- RunnableBranch is a LangChain Runnable used to create conditional workflows. It evaluates conditions against the current input and executes the first matching Runnable; if no condition matches, it executes the default Runnable.